# 01 — Create Sample Order Batches

**Project:** Orchestrated Incremental Orders Pipeline with Databricks Workflows  
**Layer:** Source Data Generation  
**Target Schema:** `orders_workflow_project`

This notebook creates sample source order batches for an orchestrated incremental pipeline.

The goal of this project is to simulate how an e-commerce order pipeline can be orchestrated with Databricks Workflows / Jobs.

This notebook creates three source batch tables:

- `orders_workflow_project.source_orders_batch_1`
- `orders_workflow_project.source_orders_batch_2`
- `orders_workflow_project.source_orders_batch_3`

These batches will later be processed by Bronze, Silver, Gold, and Validation notebooks inside a Databricks Workflow.

In [0]:
dbutils.widgets.text("pipeline_run_id", "manual_run", "Pipeline Run ID")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print("Pipeline Run ID: {pipeline_run_id}")

In [0]:
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, IntegerType
)
from pyspark.sql.functions import current_timestamp, lit

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS orders_workflow_project")
spark.sql("USE SCHEMA orders_workflow_project")

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS orders_workflow_project.pipeline_run_audit (
    pipeline_run_id STRING,
    task_name STRING,
    status STRING,
    records_processed BIGINT,
    message STRING,
    processed_ts TIMESTAMP
)""")

In [0]:
order_event_schema = StructType([
    StructField("batch_id", IntegerType(), False),
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_amount", DoubleType(), False),
    StructField("event_ts", TimestampType(), False),
    StructField("source_system", StringType(), False)
])

In [0]:
batch_1_data = [
    (1, "1001", "C001", "pending", 250.00, datetime(2026, 6, 1, 10, 0, 0), "ecommerce_orders"),
    (1, "1002", "C002", "pending", 125.50, datetime(2026, 6, 1, 10, 5, 0), "ecommerce_orders"),
    (1, "1003", "C003", "pending", 780.90, datetime(2026, 6, 1, 10, 10, 0), "ecommerce_orders")
]

batch_1_df = spark.createDataFrame(batch_1_data, order_event_schema)

display(batch_1_df)

In [0]:
batch_2_data = [
    (2, "1001", "C001", "shipped", 250.00, datetime(2026, 6, 1, 12, 0, 0), "ecommerce_orders"),
    (2, "1002", "C002", "cancelled", 125.50, datetime(2026, 6, 1, 12, 15, 0), "ecommerce_orders"),
    (2, "1004", "C004", "pending", 310.75, datetime(2026, 6, 1, 12, 30, 0), "ecommerce_orders")
]

batch_2_df = spark.createDataFrame(batch_2_data, order_event_schema)

display(batch_2_df)

In [0]:
batch_3_data = [
    (3, "1001", "C001", "delivered", 250.00, datetime(2026, 6, 2, 9, 0, 0), "ecommerce_orders"),
    (3, "1003", "C003", "shipped", 780.90, datetime(2026, 6, 2, 9, 15, 0), "ecommerce_orders"),
    (3, "1004", "C004", "delivered", 310.75, datetime(2026, 6, 2, 9, 30, 0), "ecommerce_orders"),
    (3, "1005", "C005", "pending", 95.25, datetime(2026, 6, 2, 9, 45, 0), "ecommerce_orders")
]

batch_3_df = spark.createDataFrame(batch_3_data, order_event_schema)

display(batch_3_df)

In [0]:
batch_1_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders_workflow_project.source_orders_batch_1")

batch_2_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders_workflow_project.source_orders_batch_2")

batch_3_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders_workflow_project.source_orders_batch_3")

print("Source batch tables created successfully.")

In [0]:
display(spark.sql("""
SHOW TABLES IN orders_workflow_project
"""))

In [0]:
source_counts_df = spark.sql("""
SELECT 'source_orders_batch_1' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.source_orders_batch_1

UNION ALL

SELECT 'source_orders_batch_2' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.source_orders_batch_2

UNION ALL

SELECT 'source_orders_batch_3' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.source_orders_batch_3
""")

display(source_counts_df)

In [0]:
total_source_records = source_counts_df.selectExpr("SUM(total_records) AS total").collect()[0]["total"]

print(f"Total source records created: {total_source_records}")

In [0]:
pipeline_run_id_sql = pipeline_run_id.replace("'", "''")
task_name_sql = "01_create_sample_batches"
status_sql = "SUCCESS"
message_sql = "Source batch tables created successfully."

spark.sql(f"""
INSERT INTO orders_workflow_project.pipeline_run_audit
SELECT
    '{pipeline_run_id_sql}' AS pipeline_run_id,
    '{task_name_sql}' AS task_name,
    '{status_sql}' AS status,
    CAST({int(total_source_records)} AS BIGINT) AS records_processed,
    '{message_sql}' AS message,
    current_timestamp() AS processed_ts
""")

print("Audit record inserted successfully.")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.pipeline_run_audit
    ORDER BY processed_ts DESC
    """)
)